# Video 01 - Exercise Solutions

Try the exercises in [exercises.md](../exercises.md) yourself first. These are **one** good answer each - yours can look different and still be right.

In [0]:
import time
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"

In [0]:
# Exercise 1: same question 3 times - the answers vary (Video 03 explains why: temperature)
for attempt in range(1, 4):
    reply = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Explain what an AI agent is to a 10-year-old, in 2 sentences."}],
    )
    print(f"Attempt {attempt}: {reply.choices[0].message.content}\n")

In [0]:
# Exercise 2: wrap the call in a function that RETURNS the text
def ask(question: str, model: str = MODEL) -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}],
    )
    return response.choices[0].message.content

for q in ["What is a Python list?", "Name one planet with rings.", "What does LLM stand for?"]:
    print(f"Q: {q}\nA: {ask(q)}\n")

In [0]:
# Exercise 3: same question, three models - compare answer, tokens and speed
MODELS = [
    "databricks-meta-llama-3-3-70b-instruct",
    "databricks-claude-haiku-4-5",
    "databricks-meta-llama-3-1-8b-instruct",   # much smaller model
]
question = "In one sentence, why would an AI agent need tools?"

for model in MODELS:
    try:
        start = time.time()
        response = client.chat.completions.create(model=model, messages=[{"role": "user", "content": question}])
        seconds = time.time() - start
        print(f"--- {model} ---")
        print(response.choices[0].message.content)
        print(f"tokens: {response.usage.total_tokens} | time: {seconds:.2f}s\n")
    except Exception as e:   # the model may not exist in every workspace
        print(f"--- {model} --- not available: {str(e)[:80]}\n")

# Typical takeaway: small models are faster and cheaper, bigger models give better answers
# on hard tasks. Pick the smallest model that does the job well.

In [0]:
# Bonus: questions the model cannot know. Watch for guessing vs refusing vs hallucinating.
for q in [
    "What is my name?",
    "What's the weather in Perth right now?",
    "How many files are in my Databricks home folder?",
]:
    print(f"Q: {q}\nA: {ask(q)}\n")

# Each of these needs a TOOL: a function that fetches the real answer. Coming in Video 06.